GarNet Lite: Qkeras vs hls4ml
=============================

In this Jupyter notebook, we take a trained QKeras GarNet lite model, convert it to hls using hls4ml and evaluate accuracies.

Load Dataset
---

In [ ]:
from utils.data import load_data

X_train, y_train, X_test, y_test = load_data(suffix='baseline_1')

# We do not need the cluster sizes here
X_train = X_train[0]
X_test = X_test[0]

Load QKeras Model
---

If the entire model was saved, we need to import GarNetLayer, regression loss and add QKeras layers

In [ ]:
from keras.models import load_model
from qkeras.utils import _add_supported_quantized_objects

from hls4ml_garnet_lite.keras_model.garnet_lite import GarNetLayer  # noqa: F401
from utils.files import model_path
from utils.training import regression_loss  # noqa: F401

model_name = "garnet_lite_8_0_pruned"

keras_model_path = str(model_path / f"{model_name}.keras")

co = {}
_add_supported_quantized_objects(co)
co['GarNetLayer'] = GarNetLayer
co['regression_loss'] = regression_loss
keras_model = load_model(keras_model_path, custom_objects=co)
keras_model.summary()

Convert QKeras Model to HLS
---

Register GarNetLayer

In [ ]:
import hls4ml
from hls4ml_garnet_lite.hls4ml_extension.garnet_lite import HGarNetLayer
from hls4ml_garnet_lite.hls4ml_extension.garnet_lite_parser import parse_garnet_layer
from hls4ml_garnet_lite.hls4ml_extension.garnet_lite_template import GarNetLayerConfigTemplate, GarNetLayerFunctionTemplate
from utils.files import get_project_root_dir

project_root_dir = get_project_root_dir("hls4ml-garnet-lite")

try:
    hls4ml.converters.register_keras_v2_layer_handler('GarNetLayer', parse_garnet_layer)
    hls4ml.model.layers.register_layer('GarNetLayer', HGarNetLayer)

    backend = hls4ml.backends.get_backend('Vitis')
    backend.register_template(GarNetLayerConfigTemplate)
    backend.register_template(GarNetLayerFunctionTemplate)
    backend.register_source(project_root_dir / "hls4ml_garnet_lite" / "hls" / "nnet_garnet_lite.h")
except:
    pass  # templates already registered

In [ ]:
import hls4ml
from utils.files import hls4ml_out_path
from utils.hls_config import set_garnet_lite_hls_config

hls_config = hls4ml.utils.config_from_keras_model(
    keras_model, granularity="name", max_precision="ap_fixed<16,8,AP_RND,AP_SAT>"
)
set_garnet_lite_hls_config(hls_config)


hls_model = hls4ml.converters.convert_from_keras_model(
    keras_model,
    hls_config=hls_config,
    backend="Vitis",
    output_dir=str(hls4ml_out_path / model_name),
)

In [ ]:
hls_model.compile()

In [ ]:
from utils.evaluation import garnet_predict

y_energy_pred, y_pid_pred = garnet_predict(keras_model, X_test)
y_energy_pred_hls, y_pid_pred_hls = garnet_predict(hls_model, X_test)

y_energy_pred *= 100
y_energy_pred_hls *= 100

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.metrics import auc, roc_curve

sns.set_theme()
sns.set_style('darkgrid')

fpr_keras, tpr_keras, thresh_keras = roc_curve(y_test[1], y_pid_pred, pos_label=1)
fpr_hls, tpr_hls, thresh_hls = roc_curve(y_test[1], y_pid_pred_hls, pos_label=1)

auc_keras = auc(1 - fpr_keras, tpr_keras)
auc_hls = auc(1 - fpr_hls, tpr_hls)

fig = plt.figure(figsize=(4, 4))
plt.plot(1 - fpr_keras, tpr_keras, label=f"Keras (AUC = {auc_keras:.3f})")
plt.plot(1 - fpr_hls, tpr_hls, label=f"HLS (AUC = {auc_hls:.3f})")
plt.ylim((0.7, 1))
plt.xlim((0.7, 1))
plt.xlabel('Pion rejection efficiency')
plt.ylabel('Electron identification efficiency')
plt.title("Classification")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

from utils.evaluation import response_boxplot

response_boxplot(
    y_test[1],
    y_test[0],
    np.reshape(y_energy_pred, (y_energy_pred.shape[0],)),
    "Keras vs Ground Truth (quant)",
    "$E_{pred}/E_{true}$",
)
response_boxplot(
    y_test[1],
    y_test[0],
    np.reshape(y_energy_pred_hls, (y_energy_pred_hls.shape[0],)),
    "HLS vs Ground Truth (quant)",
    "$E_{hls}/E_{true}$",
)
response_boxplot(
    y_test[1],
    y_energy_pred,
    y_energy_pred_hls,
    "HLS vs Keras (quant)",
    "$E_{hls}/E_{keras}$",
)